# Validacao do conteudo do L2

### Validacao do spot 24x7

✔ Index é DatetimeIndex

✔ Timezone = UTC

✔ Ordenado crescente

✔ Sem timestamps duplicados

✔ Frequência exatamente 1 dia (24/7)

In [1]:
import os
import pandas as pd
from pathlib import Path

BASE_PATH = Path("/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/02_intermediate/spot/daily")

ONE_DAY = pd.Timedelta(days=1)

def validate_asset(path: Path):
    df = pd.read_parquet(path)

    errors = []

    # Index checks
    if not isinstance(df.index, pd.DatetimeIndex):
        errors.append("Index is not DatetimeIndex")
    else:
        if df.index.tz is None or str(df.index.tz) != "UTC":
            errors.append("Index timezone is not UTC")

        if not df.index.is_monotonic_increasing:
            errors.append("Index not monotonic increasing")

        if df.index.duplicated().any():
            errors.append("Duplicate timestamps")

        if len(df) > 1:
            diffs = df.index.to_series().diff().dropna()
            if not (diffs == ONE_DAY).all():
                errors.append("Frequency is not exactly 1 day")

    # Required columns
    required_cols = ["open", "high", "low", "close", "volume", "trades", "close_time"]
    for col in required_cols:
        if col not in df.columns:
            errors.append(f"Missing column: {col}")

    # Dtypes
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns and df[col].dtype != "float64":
            errors.append(f"{col} not float64")

    if "trades" in df.columns and df["trades"].dtype != "int64":
        errors.append("trades not int64")

    # NaN check
    if df[required_cols].isna().any().any():
        errors.append("NaN in required columns")

    # OHLC integrity
    if not ((df["high"] >= df[["open", "close"]].max(axis=1)).all()):
        errors.append("High integrity violated")

    if not ((df["low"] <= df[["open", "close"]].min(axis=1)).all()):
        errors.append("Low integrity violated")

    if not (df["high"] >= df["low"]).all():
        errors.append("High < Low detected")

    if not (df[["open","high","low","close"]] > 0).all().all():
        errors.append("Non-positive price detected")

    # Volume integrity
    if (df["volume"] < 0).any():
        errors.append("Negative volume detected")

    if (df["trades"] < 0).any():
        errors.append("Negative trades detected")

    return errors


print("\n===== L2 SPOT VALIDATION =====\n")

for file in sorted(BASE_PATH.glob("*.parquet")):
    errors = validate_asset(file)
    asset = file.stem

    if errors:
        print(f"❌ {asset}")
        for e in errors:
            print("   -", e)
    else:
        print(f"✅ {asset} OK")

print("\nValidation completed.")


===== L2 SPOT VALIDATION =====

✅ ADAUSDT OK
✅ AVAXUSDT OK
✅ BNBUSDT OK
✅ BTCUSDT OK
✅ ETHUSDT OK
✅ LINKUSDT OK
✅ SOLUSDT OK
✅ XRPUSDT OK

Validation completed.


### validacao bd

In [2]:
import pandas as pd
from pathlib import Path

BASE_PATH = Path(
    "/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/02_intermediate/spot/business_day"
)

REQUIRED_COLUMNS = {"open", "high", "low", "close", "volume"}
ONE_DAY = pd.Timedelta(days=1)
THREE_DAYS = pd.Timedelta(days=3)

print("\n===== L2 SPOT BUSINESS DAY VALIDATION =====")

for file in sorted(BASE_PATH.glob("*.parquet")):
    name = file.stem
    df = pd.read_parquet(file)

    try:
        # --- Index validation ---
        if not isinstance(df.index, pd.DatetimeIndex):
            raise ValueError("Index is not DatetimeIndex.")

        if df.index.tz is None or str(df.index.tz) != "UTC":
            raise ValueError("Index is not UTC.")

        if df.index.duplicated().any():
            raise ValueError("Duplicate timestamps found.")

        if not df.index.is_monotonic_increasing:
            raise ValueError("Index not sorted ascending.")

        # --- Required columns ---
        missing = REQUIRED_COLUMNS - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        # --- NaN check ---
        if df[list(REQUIRED_COLUMNS)].isna().any().any():
            raise ValueError("NaN detected in required columns.")

        # --- Object column check ---
        if df.select_dtypes(include=["object"]).shape[1] > 0:
            raise ValueError("Object dtype columns detected.")

        # --- Business day frequency ---
        if len(df) > 1:
            diffs = df.index.to_series().diff().dropna()
            valid = (diffs == ONE_DAY) | (diffs == THREE_DAYS)
            if not valid.all():
                raise ValueError("Invalid business-day gaps detected.")

        # --- OHLC integrity ---
        o, h, l, c = df["open"], df["high"], df["low"], df["close"]

        if ((h < o) | (h < c)).any():
            raise ValueError("High < max(open, close).")

        if ((l > o) | (l > c)).any():
            raise ValueError("Low > min(open, close).")

        if (h < l).any():
            raise ValueError("High < Low.")

        if (o <= 0).any() or (h <= 0).any() or (l <= 0).any() or (c <= 0).any():
            raise ValueError("Non-positive OHLC value detected.")

        if (df["volume"] < 0).any():
            raise ValueError("Negative volume detected.")

        print(f"✅ {name} OK")

    except Exception as e:
        print(f"❌ {name} FAILED → {e}")


===== L2 SPOT BUSINESS DAY VALIDATION =====
✅ gold OK
✅ nasdaq OK
✅ sp500 OK


### analise fred

In [3]:
import pandas as pd
import numpy as np

path = "/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/01_raw/macro/daily/DGS2.parquet"

print("===== LOADING FILE =====")
df = pd.read_parquet(path)

print("\n===== BASIC INFO =====")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)

print("\n===== HEAD =====")
print(df.head())

print("\n===== TAIL =====")
print(df.tail())

print("\n===== DATE ANALYSIS =====")
if "date" in df.columns:
    date_col = "date"
elif "timestamp" in df.columns:
    date_col = "timestamp"
else:
    date_col = None

if date_col:
    print("Min date:", df[date_col].min())
    print("Max date:", df[date_col].max())
    print("Timezone:", df[date_col].dt.tz)
    print("Duplicated dates:", df[date_col].duplicated().sum())
else:
    print("No date/timestamp column found.")

print("\n===== NUMERIC SUMMARY =====")
print(df.describe(include="all"))

print("\n===== NULL CHECK =====")
print(df.isna().sum())

print("\n===== OBJECT COLUMNS =====")
print(df.select_dtypes(include=["object"]).columns.tolist())

===== LOADING FILE =====

===== BASIC INFO =====
Shape: (1406, 2)
Columns: ['date', 'value']

Dtypes:
date     datetime64[ns, UTC]
value                float64
dtype: object

===== HEAD =====
                       date  value
0 2020-10-01 00:00:00+00:00   0.14
1 2020-10-02 00:00:00+00:00   0.13
2 2020-10-05 00:00:00+00:00   0.14
3 2020-10-06 00:00:00+00:00   0.14
4 2020-10-07 00:00:00+00:00   0.16

===== TAIL =====
                          date  value
1401 2026-02-13 00:00:00+00:00   3.40
1402 2026-02-16 00:00:00+00:00    NaN
1403 2026-02-17 00:00:00+00:00   3.43
1404 2026-02-18 00:00:00+00:00   3.47
1405 2026-02-19 00:00:00+00:00   3.47

===== DATE ANALYSIS =====
Min date: 2020-10-01 00:00:00+00:00
Max date: 2026-02-19 00:00:00+00:00
Timezone: UTC
Duplicated dates: 0

===== NUMERIC SUMMARY =====
                                      date        value
count                                 1406  1344.000000
mean   2023-06-11 21:35:35.419630336+00:00     3.068028
min              2020-

### vix L1

In [4]:
import pandas as pd
import numpy as np

path = "/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/01_raw/macro/daily/vix.parquet"

print("===== LOADING FILE =====")
df = pd.read_parquet(path)

print("\n===== BASIC INFO =====")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)

print("\n===== HEAD =====")
print(df.head())

print("\n===== TAIL =====")
print(df.tail())

print("\n===== DATE ANALYSIS =====")
if "date" in df.columns:
    date_col = "date"
elif "timestamp" in df.columns:
    date_col = "timestamp"
else:
    date_col = None

if date_col:
    print("Min date:", df[date_col].min())
    print("Max date:", df[date_col].max())
    print("Timezone:", df[date_col].dt.tz)
    print("Duplicated dates:", df[date_col].duplicated().sum())
else:
    print("No date/timestamp column found.")

print("\n===== NUMERIC SUMMARY =====")
print(df.describe(include="all"))

print("\n===== NULL CHECK =====")
print(df.isna().sum())

print("\n===== OBJECT COLUMNS =====")
print(df.select_dtypes(include=["object"]).columns.tolist())

===== LOADING FILE =====

===== BASIC INFO =====
Shape: (1969, 6)
Columns: ['date', 'close', 'high', 'low', 'open', 'volume']

Dtypes:
date      datetime64[ns, UTC]
close                 float64
high                  float64
low                   float64
open                  float64
volume                  int64
dtype: object

===== HEAD =====
                       date      close       high    low       open  volume
0 2020-10-01 00:00:00+00:00  26.700001  27.110001  25.33  25.780001       0
1 2020-10-02 00:00:00+00:00  27.629999  29.900000  26.93  28.870001       0
2 2020-10-03 00:00:00+00:00  27.629999  29.900000  26.93  28.870001       0
3 2020-10-04 00:00:00+00:00  27.629999  29.900000  26.93  28.870001       0
4 2020-10-05 00:00:00+00:00  27.959999  29.690001  27.27  29.520000       0

===== TAIL =====
                          date      close       high    low       open  volume
1964 2026-02-16 00:00:00+00:00  20.600000  22.400000  18.92  21.480000       0
1965 2026-02-17 00:00